In [1]:
import os 
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sqlalchemy.exc import SQLAlchemyError

# Makine Öğrenmesi İçin Kullanacağımız kütüphaneler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,r2_score

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

load_dotenv()

def create_db_engine():
    try:
        engine = create_engine(
            f"postgresql://{os.getenv('POSTGRES_USER')}:"
            f"{os.getenv('POSTGRES_PASSWORD')}@localhost:"
            f"{os.getenv('POSTGRES_PORT')}/"
            f"{os.getenv('POSTGRES_DB')}"
        )

        with engine.connect() as conn:
            print("Bağlantı Başarılı!")

        return engine

    except SQLAlchemyError as e:
        print("SQLAlchemy hatası:", e)
        return None


def get_data(query):
    engine = create_db_engine()

    if engine is None:
        return None

    try:
        df = pd.read_sql(query, engine)
        print("Veri çekildi")
        return df

    except SQLAlchemyError as e:
        print("Query hatası:", e)
        return None

    except Exception as e:
        print("Genel hata:", e)
        return None


query = """
SELECT 
    f.rent_euro,
    l.is_dublin,
    p.property_type,
    p.bedrooms_num,
    t.rent_year
FROM gold.fact_rent f
JOIN gold.dim_location l ON f.dim_location_id = l.id
JOIN gold.dim_property p ON f.dim_property_id = p.id
JOIN gold.dim_time t ON f.dim_time_id = t.id
"""

df = get_data(query)

if df is not None:
    print(df.head())
    print(df.describe())

Bağlantı Başarılı!
Veri çekildi
   rent_euro  is_dublin       property_type  bedrooms_num  rent_year
0     835.90      False  all property types             0       2020
1     860.74      False  all property types             0       2020
2     910.91      False  all property types             0       2020
3     812.14      False  all property types             0       2020
4     666.31      False  all property types             0       2020
          rent_euro  bedrooms_num     rent_year
count  50208.000000  50208.000000  50208.000000
mean    1306.508984      1.678697   2021.950506
std      528.333082      1.146409      1.559695
min      401.580000      0.000000   2020.000000
25%      888.455000      1.000000   2021.000000
50%     1200.435000      2.000000   2022.000000
75%     1649.057500      2.000000   2023.000000
max     5372.870000      4.000000   2025.000000


In [5]:
df_ml = pd.get_dummies(df,columns = ["property_type"],drop_first = True )

print(f"Boş Veri Sayısı:\n{df_ml.isnull().sum()}")
print(f"Yeni Veri Seti (İlk 5 Satır): {df_ml.head()}")

Boş Veri Sayısı:
rent_euro                            0
is_dublin                            0
bedrooms_num                         0
rent_year                            0
property_type_apartment              0
property_type_detached house         0
property_type_other flats            0
property_type_semi detached house    0
property_type_terrace house          0
dtype: int64
Yeni Veri Seti (İlk 5 Satır):    rent_euro  is_dublin  bedrooms_num  rent_year  property_type_apartment  \
0     835.90      False             0       2020                    False   
1     860.74      False             0       2020                    False   
2     910.91      False             0       2020                    False   
3     812.14      False             0       2020                    False   
4     666.31      False             0       2020                    False   

   property_type_detached house  property_type_other flats  \
0                         False                      False   
1 

In [7]:
# Hedef değişkenimiz (tahmin etmek istediğimiz): rent_euro
# ADIM 2: Soruları (X) ve Cevabı (y) ayır

# y = Tahmin etmek istediğimiz şey (Kira fiyatı)
y = df_ml['rent_euro']

# X = Tahminde kullanacağımız her şey (rent_euro hariç)
X = df_ml.drop('rent_euro', axis=1)

print(f"X (özellikler) boyutu: {X.shape}  →  {X.shape[0]} satır, {X.shape[1]} sütun")
print(f"y (hedef) boyutu: {y.shape}  →  {y.shape[0]} kira fiyatı")
print("\nX sütunları:", X.columns.tolist())



X (özellikler) boyutu: (50208, 8)  →  50208 satır, 8 sütun
y (hedef) boyutu: (50208,)  →  50208 kira fiyatı

X sütunları: ['is_dublin', 'bedrooms_num', 'rent_year', 'property_type_apartment', 'property_type_detached house', 'property_type_other flats', 'property_type_semi detached house', 'property_type_terrace house']


In [8]:
# ADIM 3: Veriyi %80 Eğitim, %20 Test olarak böl

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,    # %20'sini sınav için ayır
    random_state=42   # Her çalıştırmada aynı bölünme olsun (42 bir "şans sayısı")
)

print(f"Eğitim seti: {len(X_train)} satır (verin %80'i)")
print(f"Test seti:   {len(X_test)} satır (verin %20'si)")


Eğitim seti: 40166 satır (verin %80'i)
Test seti:   10042 satır (verin %20'si)


In [9]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

print("Model eğitimi başlıyor... (birkaç saniye sürebilir)")

model.fit(X_train,y_train)

print("✅ Model başarıyla eğitildi!")

Model eğitimi başlıyor... (birkaç saniye sürebilir)
✅ Model başarıyla eğitildi!


In [10]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test,y_pred)

r2 = r2_score(y_test,y_pred)

print("📊 MODEL SONUÇLARI:")
print(f"  Ortalama Hata (MAE): ±{mae:.2f} Euro")
print(f"  Başarı Oranı (R²):    %{r2*100:.2f}")
print()
print("💡 MAE ne anlama geliyor?")
print(f"   Model, gerçek kira fiyatından ortalama ±{mae:.0f} Euro yanılıyor.")
print()
print("💡 R² ne anlama geliyor?")
print(f"   Modelin tahminleri, gerçek değerlerin %{r2*100:.0f}'ini açıklıyor.")
print(f"   100'e ne kadar yakınsa o kadar iyi! (0.80+ iyi, 0.90+ mükemmel)")

📊 MODEL SONUÇLARI:
  Ortalama Hata (MAE): ±215.13 Euro
  Başarı Oranı (R²):    %72.75

💡 MAE ne anlama geliyor?
   Model, gerçek kira fiyatından ortalama ±215 Euro yanılıyor.

💡 R² ne anlama geliyor?
   Modelin tahminleri, gerçek değerlerin %73'ini açıklıyor.
   100'e ne kadar yakınsa o kadar iyi! (0.80+ iyi, 0.90+ mükemmel)


In [12]:
model_v2 = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    random_state=42    
)

print("Geliştirilmiş model eğitiliyor... (biraz daha sürebilir)")
model_v2.fit(X_train, y_train)

y_pred_v2 = model_v2.predict(X_test)   # ← model_v2 olmalı!

mae_v2 = mean_absolute_error(y_test, y_pred_v2)
r2_v2  = r2_score(y_test, y_pred_v2)

print("\n📊 GELİŞTİRİLMİŞ MODEL SONUÇLARI:")
print(f"  Ortalama Hata (MAE): ±{mae_v2:.2f} Euro  (önceki: ±{mae:.2f} Euro)")
print(f"  Başarı Oranı (R²):    %{r2_v2*100:.2f}         (önceki: %{r2*100:.2f})")

if mae_v2 < mae:
    print(f"\n✅ Model iyileşti! Hata {mae - mae_v2:.2f} Euro azaldı.")
else:
    print("\n⚠️ Bu parametreler daha iyi sonuç vermedi.")



Geliştirilmiş model eğitiliyor... (biraz daha sürebilir)

📊 GELİŞTİRİLMİŞ MODEL SONUÇLARI:
  Ortalama Hata (MAE): ±215.07 Euro  (önceki: ±215.13 Euro)
  Başarı Oranı (R²):    %72.76         (önceki: %72.75)

✅ Model iyileşti! Hata 0.06 Euro azaldı.


In [13]:
# ADIM 7: Yapay Zekaya Kira Sorusu Sor!

# Önce modelin beklediği sütunları hatırlayalım
print("Modelin beklediği sütunlar:")
print(X.columns.tolist())


Modelin beklediği sütunlar:
['is_dublin', 'bedrooms_num', 'rent_year', 'property_type_apartment', 'property_type_detached house', 'property_type_other flats', 'property_type_semi detached house', 'property_type_terrace house']


In [14]:
# ADIM 8: Yapay Zekaya Kira Sorusu Sor!

def kira_tahmin_et(yil, oda_sayisi, dublin_mi, ev_tipi):
    """
    yil         : 2020-2025 arası bir yıl
    oda_sayisi  : 0, 1, 2, 3 veya 4
    dublin_mi   : True veya False
    ev_tipi     : "apartment", "detached house", "semi detached house",
                  "terrace house", "other flats", "all" 
    """
    
    # Boş bir satır oluştur (tüm ev tipleri 0 başlar)
    veri = {
        'is_dublin':                            [dublin_mi],
        'bedrooms_num':                          [oda_sayisi],
        'rent_year':                             [yil],
        'property_type_apartment':               [0],
        'property_type_detached house':          [0],
        'property_type_other flats':             [0],
        'property_type_semi detached house':     [0],
        'property_type_terrace house':           [0],
    }
    
    # Seçilen ev tipini 1 yap
    if ev_tipi != "all":
        sutun_adi = f'property_type_{ev_tipi}'
        if sutun_adi in veri:
            veri[sutun_adi] = [1]
    
    df_tahmin = pd.DataFrame(veri)
    tahmin = model_v2.predict(df_tahmin)[0]
    
    print(f"🏠 {yil} yılında, {'Dublin' if dublin_mi else 'Dublin dışı'}, "
          f"{oda_sayisi} odalı, {ev_tipi}")
    print(f"💶 Tahmini Kira: {tahmin:.2f} Euro/ay")
    print("-" * 50)
    return tahmin


# --- SORULAR ---

kira_tahmin_et(yil=2025, oda_sayisi=3, dublin_mi=True,  ev_tipi="apartment")
kira_tahmin_et(yil=2025, oda_sayisi=3, dublin_mi=False, ev_tipi="apartment")
kira_tahmin_et(yil=2020, oda_sayisi=1, dublin_mi=True,  ev_tipi="apartment")
kira_tahmin_et(yil=2025, oda_sayisi=4, dublin_mi=True,  ev_tipi="detached house")


🏠 2025 yılında, Dublin, 3 odalı, apartment
💶 Tahmini Kira: 2890.99 Euro/ay
--------------------------------------------------
🏠 2025 yılında, Dublin dışı, 3 odalı, apartment
💶 Tahmini Kira: 1651.23 Euro/ay
--------------------------------------------------
🏠 2020 yılında, Dublin, 1 odalı, apartment
💶 Tahmini Kira: 1428.98 Euro/ay
--------------------------------------------------
🏠 2025 yılında, Dublin, 4 odalı, detached house
💶 Tahmini Kira: 3929.43 Euro/ay
--------------------------------------------------


3929.425096085675